In [1]:
# ==========================
# STEP 1 : INSTALL LIBRARIES
# ==========================
!pip install -q fastapi uvicorn pyngrok sentence-transformers transformers torch nltk

In [2]:
# ==========================
# STEP 2 : IMPORT LIBRARIES
# ==========================
import pandas as pd
import numpy as np
import nltk
import re

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
import os
os.listdir()

['.config',
 'Software Questions.csv',
 'job_description.csv',
 'resume_data.csv',
 'LinkedIn_RDB_three.csv',
 'sample_data']

In [5]:
resume_df = pd.read_csv('resume_data.csv')
jobs_df = pd.read_csv('LinkedIn_RDB_three.csv')
jd_df = pd.read_csv('job_description.csv')

# Software Questions file encoding issue handle
try:
    questions_df = pd.read_csv('Software Questions.csv')
except UnicodeDecodeError:
    questions_df = pd.read_csv('Software Questions.csv', encoding='latin1')

print("Resume Dataset :", resume_df.shape)
print("Jobs Dataset :", jobs_df.shape)
print("JD Dataset :", jd_df.shape)
print("Questions Dataset :", questions_df.shape)

Resume Dataset : (9544, 35)
Jobs Dataset : (6328, 15)
JD Dataset : (325, 6)
Questions Dataset : (200, 5)


In [6]:
# Check Columns
print("Resume Columns")
print(resume_df.columns)

print("\nJobs Columns")
print(jobs_df.columns)

print("\nJD Columns")
print(jd_df.columns)

print("\nQuestions Columns")
print(questions_df.columns)

Resume Columns
Index(['address', 'career_objective', 'skills', 'educational_institution_name',
       'degree_names', 'passing_years', 'educational_results', 'result_types',
       'major_field_of_studies', 'professional_company_names', 'company_urls',
       'start_dates', 'end_dates', 'related_skils_in_job', 'positions',
       'locations', 'responsibilities', 'extra_curricular_activity_types',
       'extra_curricular_organization_names',
       'extra_curricular_organization_links', 'role_positions', 'languages',
       'proficiency_levels', 'certification_providers', 'certification_skills',
       'online_links', 'issue_dates', 'expiry_dates', '﻿job_position_name',
       'educationaL_requirements', 'experiencere_requirement',
       'age_requirement', 'responsibilities.1', 'skills_required',
       'matched_score'],
      dtype='object')

Jobs Columns
Index(['job_id', 'title', 'description', 'pay_period', 'work_type',
       'job_location', 'applies', 'remote_allowed', 'views', '

In [7]:
# Create Text Columns for Each Dataset

resume_df = resume_df.fillna('')
jobs_df = jobs_df.fillna('')
jd_df = jd_df.fillna('')
questions_df = questions_df.fillna('')

# Resume Text
resume_df['resume_text'] = (
    resume_df['career_objective'].astype(str) + ' ' +
    resume_df['skills'].astype(str) + ' ' +
    resume_df['degree_names'].astype(str) + ' ' +
    resume_df['major_field_of_studies'].astype(str) + ' ' +
    resume_df['related_skils_in_job'].astype(str) + ' ' +
    resume_df['skills_required'].astype(str)
)

# Job Text
jobs_df['job_text'] = (
    jobs_df['title'].astype(str) + ' ' +
    jobs_df['description'].astype(str) + ' ' +
    jobs_df['job_domain'].astype(str) + ' ' +
    jobs_df['level'].astype(str)
)

# Job Description Text
jd_df['jd_text'] = (
    jd_df['Category'].astype(str) + ' ' +
    jd_df['Description'].astype(str) + ' ' +
    jd_df['Requirement'].astype(str) + ' ' +
    jd_df['Requirements'].astype(str)
)

print("Text Columns Created Successfully")

Text Columns Created Successfully


In [8]:
# Install Sentence-BERT (Run only once)
!pip install -q sentence-transformers

In [9]:
# Load Sentence-BERT Model
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

print("Model Loaded Successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model Loaded Successfully


In [10]:
# Create Embeddings

resume_embeddings = model.encode(
    resume_df['resume_text'].tolist(),
    show_progress_bar=True
)

job_embeddings = model.encode(
    jobs_df['job_text'].tolist(),
    show_progress_bar=True
)

jd_embeddings = model.encode(
    jd_df['jd_text'].tolist(),
    show_progress_bar=True
)

print("Embeddings Created Successfully")

Batches:   0%|          | 0/299 [00:00<?, ?it/s]

Batches:   0%|          | 0/198 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embeddings Created Successfully


In [11]:
# ATS Score Function
from sklearn.metrics.pairwise import cosine_similarity

def get_ats_score(resume_index, jd_index):
    score = cosine_similarity(
        [resume_embeddings[resume_index]],
        [jd_embeddings[jd_index]]
    )[0][0]

    return round(score * 100, 2)

In [12]:
# Example
score = get_ats_score(0, 0)
print("ATS Score :", score, "%")

ATS Score : 32.24 %


In [13]:
# Resume Improvement Suggestions

def get_suggestions(resume_index, jd_index):
    resume_text = resume_df.loc[resume_index, 'resume_text'].lower()
    jd_text = jd_df.loc[jd_index, 'jd_text'].lower()

    resume_words = set(resume_text.split())
    jd_words = set(jd_text.split())

    missing_skills = jd_words - resume_words

    return list(missing_skills)[:10]

In [14]:
# Example
suggestions = get_suggestions(0, 0)
print("Suggested Skills:")
print(suggestions)

Suggested Skills:
['junior', '101', 'qualifications', 'successfully', 'analysts', 'title:', 'travel', 'great', 'engagement', 'approaches']


In [15]:
# Job Recommendation Function

def recommend_jobs(resume_index, top_n=5):
    scores = cosine_similarity(
        [resume_embeddings[resume_index]],
        job_embeddings
    )[0]

    top_indices = scores.argsort()[-top_n:][::-1]

    recommendations = jobs_df.iloc[
        top_indices
    ][['title', 'job_location', 'job_domain']]

    return recommendations

In [16]:
# Example
recommend_jobs(0)

,title,job_location,job_domain
4578,Cloud Engineer: 23-02161,"San Jose, CA",Information Technology
1066,Big Data Developer- W2 Contract Only,"Woodbridge, NJ",Engineering
127,Senior Data Engineer,"Virginia, United States",Information Technology
462,Senior Data Engineer,"Chicago, IL",Information Technology
464,Senior Data Engineer,"Atlanta, GA",Information Technology


In [17]:
# ATS Score + Suggestions + Job Recommendation (Single Function)

def career_assistant(resume_index=0, jd_index=0, top_n=5):

    # ATS Score
    ats_score = round(
        cosine_similarity(
            [resume_embeddings[resume_index]],
            [jd_embeddings[jd_index]]
        )[0][0] * 100,
        2
    )

    # Missing Skills
    resume_words = set(
        resume_df.loc[resume_index, 'resume_text'].lower().split()
    )

    jd_words = set(
        jd_df.loc[jd_index, 'jd_text'].lower().split()
    )

    suggestions = list(jd_words - resume_words)[:10]

    # Job Recommendation
    scores = cosine_similarity(
        [resume_embeddings[resume_index]],
        job_embeddings
    )[0]

    top_indices = scores.argsort()[-top_n:][::-1]

    recommendations = jobs_df.iloc[
        top_indices
    ][['title', 'job_location', 'job_domain']]

    return ats_score, suggestions, recommendations

In [18]:
# Example

ats_score, suggestions, recommendations = career_assistant(
    resume_index=0,
    jd_index=0,
    top_n=5
)

print("ATS Score :", ats_score, "%")

print("\nSuggested Skills:")
print(suggestions)

print("\nRecommended Jobs:")
display(recommendations)

ATS Score : 32.24 %

Suggested Skills:
['junior', '101', 'qualifications', 'successfully', 'analysts', 'title:', 'travel', 'great', 'engagement', 'approaches']

Recommended Jobs:


,title,job_location,job_domain
4578,Cloud Engineer: 23-02161,"San Jose, CA",Information Technology
1066,Big Data Developer- W2 Contract Only,"Woodbridge, NJ",Engineering
127,Senior Data Engineer,"Virginia, United States",Information Technology
462,Senior Data Engineer,"Chicago, IL",Information Technology
464,Senior Data Engineer,"Atlanta, GA",Information Technology


In [19]:
# Example: Resume 0 vs Job 0
resume_text = resume_df.loc[0, 'resume_text']
job_text = jobs_df.loc[0, 'job_text']

score = round(
    cosine_similarity(
        [model.encode(resume_text)],
        [model.encode(job_text)]
    )[0][0] * 100,
    2
)

print("ATS Score :", score, "%")

ATS Score : 29.26 %


In [20]:
scores = cosine_similarity(
    [resume_embeddings[0]],
    job_embeddings
)[0]

best_job_index = scores.argmax()

print("Best Job Index:", best_job_index)
print("Best Match Score:", round(scores[best_job_index] * 100, 2), "%")

Best Job Index: 4578
Best Match Score: 75.75 %


In [21]:
ats_score = round(scores[best_job_index] * 100, 2)
print("ATS Score:", ats_score, "%")

ATS Score: 75.75 %


In [22]:
best_job = jobs_df.iloc[best_job_index]

print("Job Title :", best_job['title'])
print("Location :", best_job['job_location'])
print("Domain :", best_job['job_domain'])
print("\nJob Description:\n")
print(best_job['description'])

Job Title : Cloud Engineer: 23-02161
Location : San Jose, CA
Domain : Information Technology

Job Description:

Primary Skills: Hadoop, Apache Spark, Java, Python, Puppet, Data Governance, Analytics, Visualization, Operation EngineerContract Type: W2 Only (US Citizen to work on a Federal project)Duration: 12 months (possible extension)Location: Remote (San Jose, CA preferred)
Grow your skills by working with industry leaders 
JOB RESPONSIBILITIESAs a Cloud Engineer of the Data Platform and Infrastructure Engineering team, you will have an outstanding opportunity to use your technical expertise and creativity in Big Data to design, develop and innovative a global data analytics platform. We are seeking a talented, ambitious, and highly creative software engineer to join our team to continue to evolve our big data platform.
 Responsibilities:Work with team to build big data platform, deliver task with good quality, establish a data platform for the entire Cisco Collaboration solution.Wor

In [23]:
# Resume Improvement Suggestions (Meaningful Skills)

import re

def get_resume_suggestions(resume_index, job_index, top_n=10):

    resume_text = resume_df.loc[resume_index, 'resume_text'].lower()
    job_text = jobs_df.loc[job_index, 'job_text'].lower()

    resume_words = set(re.findall(r'\b[a-zA-Z][a-zA-Z+\-#.]{1,20}\b', resume_text))
    job_words = set(re.findall(r'\b[a-zA-Z][a-zA-Z+\-#.]{1,20}\b', job_text))

    stop_words = {
        'and','the','for','with','from','that','this','have','will','who',
        'our','your','their','into','such','using','work','works','team',
        'good','years','year','preferred','experience','requirements',
        'responsibilities','including','understanding','knowledge',
        'backgrounds','across','global','engineer','engineering','computer',
        'science','bachelor'
    }

    missing_skills = sorted(
        list((job_words - resume_words) - stop_words)
    )

    return missing_skills[:top_n]


suggestions = get_resume_suggestions(0, best_job_index)
print("Suggested Skills:")
print(suggestions)

Suggested Skills:
['about', 'administration', 'akraya', 'akrayaakraya', 'allowing', 'ambitious', 'an', 'analysts', 'apache', 'architecture']


In [24]:
# Skill Dictionary
skill_keywords = [
    'python','java','sql','hadoop','spark','kafka','jenkins',
    'docker','kubernetes','aws','azure','gcp','pytorch',
    'tensorflow','machine learning','deep learning','nlp',
    'data analytics','data visualization','data governance',
    'hdfs','elk','puppet','chef','git','linux'
]

def get_resume_suggestions(resume_index, job_index):

    resume_text = resume_df.loc[resume_index, 'resume_text'].lower()
    job_text = jobs_df.loc[job_index, 'job_text'].lower()

    missing_skills = []

    for skill in skill_keywords:
        if skill in job_text and skill not in resume_text:
            missing_skills.append(skill)

    return missing_skills


suggestions = get_resume_suggestions(0, best_job_index)
print("Suggested Skills:")
print(suggestions)

Suggested Skills:
['kafka', 'jenkins', 'data visualization', 'data governance', 'elk', 'puppet', 'chef']


In [25]:
def generate_mock_interview(job_index, n_questions=10):

    job_text = jobs_df.loc[job_index, 'job_text'].lower()

    selected_questions = pd.DataFrame()

    for skill in skill_keywords:
        if skill in job_text:

            temp = questions_df[
                questions_df.astype(str)
                .apply(lambda x: x.str.lower())
                .apply(lambda x: x.str.contains(skill))
                .any(axis=1)
            ]

            selected_questions = pd.concat(
                [selected_questions, temp]
            )

    selected_questions = (
        selected_questions
        .drop_duplicates()
        .head(n_questions)
    )

    return selected_questions[
        ['Question', 'Category', 'Difficulty']
    ]

In [26]:
mock_questions = generate_mock_interview(best_job_index)
display(mock_questions)

,Question,Category,Difficulty
25,What are the differences between Python 2 and ...,Languages and Frameworks,Medium
20,What is the difference between Java and JavaSc...,Languages and Frameworks,Medium
23,"Explain the use of ""this"" keyword in JavaScript.",Languages and Frameworks,Medium
27,Explain the concept of multi-threading in Java.,Languages and Frameworks,Medium
100,Explain the concept of 'closure' in JavaScript.,Front-end,Hard
110,Explain the difference between '==' and '===' ...,Front-end,Hard
123,Explain event delegation in JavaScript.,Front-end,Medium
124,Explain the concept of 'closure' in JavaScript.,Front-end,Hard
134,Explain the difference between '==' and '===' ...,Front-end,Easy
147,Explain event delegation in JavaScript.,Front-end,Medium


In [27]:
questions_df.head()

,Question Number,Question,Answer,Category,Difficulty
0,1,What is the difference between compilation and...,Compilation translates source code into machin...,General Programming,Medium
1,2,Explain the concept of polymorphism.,Polymorphism allows objects of different class...,General Programming,Medium
2,3,Define encapsulation and give an example.,Encapsulation bundles data and methods in a cl...,General Programming,Hard
3,4,"What is an abstract class, and how is it diffe...",An abstract class can't be instantiated and ca...,General Programming,Medium
4,5,Describe the principles of Object-Oriented Pro...,"OOP principles include encapsulation, inherita...",General Programming,Medium


In [28]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio

In [29]:
from fastapi import FastAPI
from pyngrok import ngrok
import nest_asyncio
import uvicorn

app = FastAPI()

In [30]:
@app.get("/")
def home():
    return {"message": "AI Career Assistant Backend Running"}

In [31]:
@app.get("/ats/{resume_index}/{job_index}")
def ats_score_api(resume_index: int, job_index: int):

    score = round(
        cosine_similarity(
            [resume_embeddings[resume_index]],
            [job_embeddings[job_index]]
        )[0][0] * 100,
        2
    )

    suggestions = get_resume_suggestions(
        resume_index,
        job_index
    )

    return {
        "ats_score": score,
        "suggestions": suggestions
    }

In [32]:
@app.get("/jobs/{resume_index}")
def recommend_jobs_api(resume_index: int):

    scores = cosine_similarity(
        [resume_embeddings[resume_index]],
        job_embeddings
    )[0]

    top_indices = scores.argsort()[-5:][::-1]

    recommendations = jobs_df.iloc[
        top_indices
    ][['title', 'job_location', 'job_domain']]

    return recommendations.to_dict(orient="records")

In [33]:
@app.get("/interview/{job_index}")
def interview_api(job_index: int):

    questions = generate_mock_interview(job_index)

    return questions.to_dict(orient="records")

In [34]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸

In [35]:
!lt --port 8000

your url is: https://ninety-mugs-nail.loca.lt
^C


In [36]:
!pip install -q nest_asyncio

In [37]:
import nest_asyncio
nest_asyncio.apply()

In [38]:
import uvicorn
server = uvicorn.Server(
    uvicorn.Config(app, host="0.0.0.0", port=8000)
)
await server.serve()

INFO:     Started server process [3408]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [3408]


In [40]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

scores = []

for i in range(len(resume_embeddings)):
    sim = cosine_similarity(
        [resume_embeddings[i]],
        job_embeddings
    )[0]

    best_score = np.max(sim) * 100
    scores.append(best_score)

overall_accuracy = np.mean(scores)

print(f"Overall Matching Accuracy: {overall_accuracy:.2f}%")

Overall Matching Accuracy: 59.20%


In [41]:
it_jobs = jobs_df[
    jobs_df['job_domain']
    .str.contains('Information Technology|Engineering',
                  case=False,
                  na=False)
].reset_index(drop=True)

it_job_embeddings = model.encode(
    it_jobs['job_text'].tolist(),
    batch_size=32,
    show_progress_bar=True
)

Batches:   0%|          | 0/37 [00:00<?, ?it/s]

In [42]:
scores = []

for i in range(len(resume_embeddings)):
    sim = cosine_similarity(
        [resume_embeddings[i]],
        it_job_embeddings
    )[0]

    scores.append(np.max(sim) * 100)

print("Average Score:", round(np.mean(scores), 2), "%")

Average Score: 56.67 %


In [1]:
!pip install pymupdf


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from fastapi import FastAPI, UploadFile, File
import fitz

In [6]:
pip install fastapi uvicorn pymupdf python-multipart

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from fastapi import FastAPI, UploadFile, File
import fitz

app = FastAPI()

In [1]:
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import fitz

app = FastAPI()

c:\Users\harsh\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
jobs_df = pd.read_csv("LinkedIn_RDB_three.csv")
jobs_df = jobs_df.fillna("")

jobs_df["job_text"] = (
    jobs_df["title"].astype(str) + " " +
    jobs_df["description"].astype(str) + " " +
    jobs_df["job_domain"].astype(str)
)

model = SentenceTransformer("all-MiniLM-L6-v2")

job_embeddings = model.encode(
    jobs_df["job_text"].tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 198/198 [10:06<00:00,  3.06s/it]


In [3]:
@app.post("/top-jobs")
async def top_jobs(file: UploadFile = File(...)):
    contents = await file.read()

    with open("temp_resume.pdf", "wb") as f:
        f.write(contents)

    doc = fitz.open("temp_resume.pdf")

    resume_text = ""

    for page in doc:
        resume_text += page.get_text()

    resume_embedding = model.encode([resume_text])

    scores = cosine_similarity(
        resume_embedding,
        job_embeddings
    )[0]

    top_indices = scores.argsort()[-5:][::-1]

    recommended_jobs = []

    for i in top_indices:
        recommended_jobs.append({
            "job_title": str(jobs_df.iloc[i]["title"]),
            "location": str(jobs_df.iloc[i]["job_location"]),
            "domain": str(jobs_df.iloc[i]["job_domain"]),
            "score": round(float(scores[i]) * 100, 2)
        })

    return {
        "top_jobs": recommended_jobs
    }

In [5]:
questions_df = pd.read_csv(
    "Software Questions.csv",
    encoding="latin1"
)

print(questions_df.columns)
print(questions_df.head())

Index(['Question Number', 'Question', 'Answer', 'Category', 'Difficulty'], dtype='str')
   Question Number                                           Question  \
0                1  What is the difference between compilation and...   
1                2               Explain the concept of polymorphism.   
2                3          Define encapsulation and give an example.   
3                4  What is an abstract class, and how is it diffe...   
4                5  Describe the principles of Object-Oriented Pro...   

                                              Answer             Category  \
0  Compilation translates source code into machin...  General Programming   
1  Polymorphism allows objects of different class...  General Programming   
2  Encapsulation bundles data and methods in a cl...  General Programming   
3  An abstract class can't be instantiated and ca...  General Programming   
4  OOP principles include encapsulation, inherita...  General Programming   

  Difficul

In [7]:
with open("Software Questions.csv", "rb") as f:
    print(f.read(500))

b'Question Number,Question,Answer,Category,Difficulty\r\n1,What is the difference between compilation and interpretation?,Compilation translates source code into machine code creating an executable file. Interpretation translates and executes code line by line without an executable.,General Programming,Medium\r\n2,Explain the concept of polymorphism.,"Polymorphism allows objects of different classes to be treated as objects of a common superclass, enabling method overriding.",General Programming,Mediu'


In [13]:
questions_df = pd.read_csv(
    "Software Questions.csv",
    encoding="latin1"
)

print(questions_df["Category"].unique())

<StringArray>
[     'General Programming',          'General Program',
          'Data Structures', 'Languages and Frameworks',
         'Database and SQL',          'Web Development',
         'Software Testing',          'Version Control',
            'System Design',                 'Security',
                   'DevOps',                'Front-end',
                 'Back-end',               'Full-stack',
               'Algorithms',         'Machine Learning',
      'Distributed Systems',               'Networking',
        'Low-level Systems',         'Database Systems',
         'Data Engineering',  'Artificial Intelligence']
Length: 22, dtype: str


In [14]:
questions_df = pd.read_csv(
    "Software Questions.csv",
    encoding="latin1"
)

In [15]:
print(questions_df.shape)
print(questions_df.head())

(200, 5)
   Question Number                                           Question  \
0                1  What is the difference between compilation and...   
1                2               Explain the concept of polymorphism.   
2                3          Define encapsulation and give an example.   
3                4  What is an abstract class, and how is it diffe...   
4                5  Describe the principles of Object-Oriented Pro...   

                                              Answer             Category  \
0  Compilation translates source code into machin...  General Programming   
1  Polymorphism allows objects of different class...  General Programming   
2  Encapsulation bundles data and methods in a cl...  General Programming   
3  An abstract class can't be instantiated and ca...  General Programming   
4  OOP principles include encapsulation, inherita...  General Programming   

  Difficulty  
0     Medium  
1     Medium  
2       Hard  
3     Medium  
4     Medium  

In [16]:
best_job = "Full Stack Engineer"

if "Full Stack" in best_job:
    categories = [
        "Front-end",
        "Back-end",
        "Full-stack",
        "Database and SQL",
        "Web Development"
    ]

elif "Machine Learning" in best_job:
    categories = [
        "Machine Learning",
        "Artificial Intelligence",
        "Data Structures",
        "Algorithms"
    ]

else:
    categories = [
        "General Programming",
        "Data Structures",
        "System Design",
        "Database Systems"
    ]

filtered = questions_df[
    questions_df["Category"].isin(categories)
]

questions = (
    filtered["Question"]
    .sample(min(5, len(filtered)))
    .tolist()
)

print(questions)

['What is CORS (Cross-Origin Resource Sharing)?', "Explain the concept of 'closure' in JavaScript.", 'What is a session in web development?', 'What is the Document Object Model (DOM)?', 'What is lazy loading in web development?']


In [17]:
best_job = "Machine Learning Engineer"

if "Full Stack" in best_job:
    categories = [
        "Front-end",
        "Back-end",
        "Full-stack",
        "Database and SQL",
        "Web Development"
    ]

elif "Machine Learning" in best_job:
    categories = [
        "Machine Learning",
        "Artificial Intelligence",
        "Data Structures",
        "Algorithms"
    ]

else:
    categories = [
        "General Programming",
        "Data Structures",
        "System Design",
        "Database Systems"
    ]

filtered = questions_df[
    questions_df["Category"].isin(categories)
]

questions = (
    filtered["Question"]
    .sample(min(5, len(filtered)))
    .tolist()
)

print(questions)

['Design and implement a concurrent hash map.', 'Develop a machine learning algorithm to detect fake news on social media.', 'Create an AI that can play a complex board game at a competitive level.', 'Implement a robust text editor with features like auto-complete and syntax highlighting.', 'What is the difference between an array and a linked list?']


In [18]:
@app.post("/mock-interview")
async def mock_interview(file: UploadFile = File(...)):
    contents = await file.read()

    with open("temp_resume.pdf", "wb") as f:
        f.write(contents)

    doc = fitz.open("temp_resume.pdf")

    resume_text = ""

    for page in doc:
        resume_text += page.get_text()

    # Resume embedding
    resume_embedding = model.encode([resume_text])

    scores = cosine_similarity(
        resume_embedding,
        job_embeddings
    )[0]

    best_job_index = int(scores.argmax())

    best_job = str(
        jobs_df.iloc[best_job_index]["title"]
    )

    # Select categories based on job
    if "Full Stack" in best_job:
        categories = [
            "Front-end",
            "Back-end",
            "Full-stack",
            "Database and SQL",
            "Web Development"
        ]

    elif "Machine Learning" in best_job:
        categories = [
            "Machine Learning",
            "Artificial Intelligence",
            "Data Structures",
            "Algorithms"
        ]

    else:
        categories = [
            "General Programming",
            "Data Structures",
            "System Design",
            "Database Systems"
        ]

    # Filter questions dataset
    filtered = questions_df[
        questions_df["Category"].isin(categories)
    ]

    # Random 5 questions
    questions = (
        filtered["Question"]
        .sample(min(5, len(filtered)))
        .tolist()
    )

    return {
        "job_title": best_job,
        "questions": questions
    }

In [19]:
questions_df = pd.read_csv(
    "Software Questions.csv",
    encoding="latin1"
)